# Task 2 — Exploratory Data Analysis (EDA)
**Bati Bank × Xente eCommerce** | Credit Risk (Alternative Data)

This notebook explores transaction-level data to understand structure, quality, distributions, correlations, and patterns that will guide feature engineering and proxy target design (RFM clustering in later tasks).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
OUTPUT_DIR = Path('analysis_outputs/task2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path('data/raw/data.csv')
df = pd.read_csv(DATA_PATH)
df['TransactionStartTime'] = pd.to_datetime(
    df['TransactionStartTime'], utc=True, errors='coerce'
)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
print(f'Unique customers: {df["CustomerId"].nunique():,}')
print(f'Date range: {df["TransactionStartTime"].min()} → {df["TransactionStartTime"].max()}')

## 1. Overview of the Data

In [ ]:
display(df.head())
display(df.dtypes.to_frame('dtype'))
print('Shape:', df.shape)

## 2. Summary Statistics

In [ ]:
numeric_cols = ['Amount', 'Value', 'CountryCode', 'PricingStrategy', 'FraudResult']
display(df[numeric_cols].describe().T)
display(df[['ProductCategory', 'ChannelId', 'CurrencyCode']].describe(include='object').T)

## 3. Distribution of Numerical Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Value'].clip(upper=df['Value'].quantile(0.99)).hist(bins=50, ax=axes[0], color='#2E86AB')
axes[0].set_title('Value (capped at 99th percentile)')
df['Amount'].clip(lower=df['Amount'].quantile(0.01), upper=df['Amount'].quantile(0.99)).hist(
    bins=50, ax=axes[1], color='#E94F37'
)
axes[1].set_title('Amount (1st–99th percentile)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'value_amount_histograms.png', dpi=120)
plt.show()

## 4. Distribution of Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = df['ProductCategory'].value_counts().head(8).index
sns.countplot(data=df, y='ProductCategory', order=order, ax=axes[0], palette='Blues_d')
axes[0].set_title('Product Category (top 8)')
sns.countplot(data=df, x='ChannelId', ax=axes[1], palette='Oranges_d')
axes[1].set_title('ChannelId')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'categorical_distributions.png', dpi=120)
plt.show()

## 5. Correlation Analysis

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=120)
plt.show()

## 6. Identifying Missing Values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct}).query('missing > 0')
if len(missing_df) == 0:
    print('No missing values detected in any column.')
else:
    display(missing_df.sort_values('pct', ascending=False))

## 7. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, y='Value', ax=axes[0])
axes[0].set_title('Value — Box Plot')
sns.boxplot(data=df, y='Amount', ax=axes[1])
axes[1].set_title('Amount — Box Plot')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'outlier_boxplots.png', dpi=120)
plt.show()
print('Extreme values exist; use winsorization/log transforms in feature engineering.')

## 8. Fraud & Temporal Patterns

In [ ]:
print('Overall fraud rate:', f"{df['FraudResult'].mean():.4%}")
fraud_by_cat = (
    df.groupby('ProductCategory')['FraudResult']
    .agg(rate='mean', n='count')
    .query('n >= 100')
    .sort_values('rate', ascending=False)
)
display(fraud_by_cat.head(8))

daily = df.set_index('TransactionStartTime').resample('D').size()
daily.plot(figsize=(10, 3), title='Daily Transaction Volume', color='#E94F37')
plt.ylabel('Transactions')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'daily_transaction_volume.png', dpi=120)
plt.show()

## 9. Top Insights (Summary)

1. **Highly imbalanced fraud signal (~0.2%)** — any classifier will need class-weighting, threshold tuning, or resampling; accuracy alone is misleading.
2. **Extreme transaction amounts** — `Value`/`Amount` are heavily right-skewed with large outliers; winsorization or log-scaling is required before RFM clustering and modeling.
3. **Category-driven risk patterns** — fraud rates differ materially by `ProductCategory` (e.g., transport and utility_bill higher than airtime), suggesting category features matter.
4. **Concentrated product mix** — most transactions are `financial_services` and `airtime`; models must generalize beyond dominant categories.
5. **Stable daily volume with short history** — ~95k transactions across ~3.7k customers from Nov 2018–Feb 2019; recency features should use a consistent snapshot date for RFM.